# Gold Layer: Logistics Business Analytics

This notebook is responsible for the final stage of the Medallion architecture — the Gold layer. 
Here, we use the cleaned and validated data from the Silver layer to create aggregated business marts. These marts directly answer the four key business questions for the Logistics profile: transit time, SLA fulfillment, delay rates, and priority order delivery speed.

In [0]:
import os
import sys
from pathlib import Path

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
workspace_path = (
    notebook_path
    if notebook_path.startswith("/Workspace/")
    else "/Workspace" + notebook_path
)
repo_root = Path(workspace_path).parent.parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.gold.logistics_marts import run_gold_marts, publish_gold_marts

marts = run_gold_marts(spark)
print(f"Gold marts successfully calculated in memory: {', '.join(marts.keys())}")

## Q1: Transit Time

**Question 1**: Evaluating the speed and predictability of delivery for each transportation type (`ship_mode`). We calculate the median and the 90th percentile (p90) of the transit time (from `ship_date` to `receipt_date`).

In [0]:
print("Q1: Transit Time Performance (Median & P90 by Ship Mode)")
display(marts["transit_time_performance"])

**Why are the fastest and most predictable transport modes different?**

- **Fastest (Median):** The median (50th percentile) shows the typical delivery time for most orders, ignoring rare extreme delays. 
- **Most predictable (P90):** The 90th percentile captures the upper bound, guaranteeing that 90% of shipments will arrive within this timeframe. 

A transport mode might have a very low median (fast), but due to frequent force majeures, its p90 could be high (unpredictable). Conversely, a slower transport mode might have a higher median but a very low spread between the median and p90, making it ideal for strict SLAs.

## Q2: Fulfillment SLA

**Question 2**: Analyzing the SLA gap between the on-time performance of individual line items versus whole orders. An order is considered fully on time only if **all** of its constituent items are delivered no later than the `commit_date`.

In [0]:
print("Q2: Order Fulfillment SLA (Line Items vs Whole Orders)")
display(marts["order_fulfillment_sla"])

**Explanation of the gap between order and line item on-time rates**

The gap between `order_fully_on_time_rate` and `lineitem_on_time_rate` occurs due to risk aggregation. Since an order is considered on time **only** if **all** of its constituent items are delivered on time, a delay in just one item from the cart causes the entire order to breach the SLA. 

Even if 95% of individual line items are delivered on time, the probability of a 5-item order arriving without delays is significantly lower. Therefore, the whole-order metric will always be lower, and this gap widens proportionally as the average number of items per customer order increases.

## Q3: Ship Mode Delays

**Question 3**: Identifying the worst-performing carriers by calculating the delay rate (`receipt_date` > `commit_date`) for each `ship_mode`.

In [0]:
print("Q3: Ship Mode Delays (Worst performing modes)")
display(marts["ship_mode_delays"])

## Q4: Priority Speed

**Question 4**: Testing the priority hypothesis: whether urgent orders (`order_priority`) are actually delivered to customers faster than regular ones.

In [0]:
print("Q4: Priority Fulfillment Speed")
display(marts["priority_fulfillment_speed"])

## Publishing Gold Marts

Once all aggregations are verified and visualized, we save them as persistent Delta tables in the target Gold schema. This will allow us to use them as a ready-made data source for the final Databricks dashboard and SLA monitoring alerts.

In [0]:
from config import env

publish_gold_marts(spark, marts)

print(f"All 4 Gold marts successfully saved to {env.CATALOG}.{env.GOLD_SCHEMA}.")

spark.sql(f"SHOW TABLES IN {env.CATALOG}.{env.GOLD_SCHEMA}").show(truncate=False)